# LAB 2 — Avaliação RAGAS: Baseline Naive RAG

**MBA em RAG & CAG Aplicados a Direito e Segurança Pública**  
**Aula 5 — Avaliação de Pipelines RAG com RAGAS**

---

## Objetivo

Calcular as **4 métricas RAGAS** no pipeline Naive RAG e registrar os resultados no **LangFuse** para rastreabilidade e comparação futura com técnicas avançadas.

## Técnica Avaliada

**#T01 — Naive RAG (Baseline)**  
Recuperação simples por kNN sem reranking, HyDE, query expansion ou outros aprimoramentos.

## Metas Mínimas do Syllabus

| Métrica | Meta Mínima | Descrição |
|---|---|---|
| **Faithfulness** | ≥ 0.80 | A resposta é factualmente sustentada pelos contextos? |
| **Answer Relevancy** | ≥ 0.75 | A resposta é relevante para a pergunta? |
| **Context Recall** | ≥ 0.70 | O contexto recuperado cobre o ground truth? |
| **Context Precision** | ≥ 0.70 | Os contextos recuperados são relevantes (sem ruído)? |

## Pré-requisitos

| Componente | Endereço | Observação |
|---|---|---|
| vLLM (Llama 3.1 8B Instruct) | `localhost:8000/v1` | LLM Judge para RAGAS |
| OpenSearch 3.x | `localhost:9200` | Índice `juridico_baseline_aula4` |
| LangFuse | `localhost:3000` | Registro de métricas |
| Dataset do LAB 1 | `dataset_avaliacao_completo.json` | Gerado no LAB 1 |

## Referência Normativa

Conforme **ABNT NBR 6023:2018**.  
ES-JOLLY, S. et al. **RAGAS: Automated Evaluation of Retrieval Augmented Generation**. arXiv:2309.15217, 2023.  
Meta. **Llama 3.1**. 2024. Disponível em: <https://llama.meta.com>. Acesso em: abr. 2026.

---
## Célula 1 — Instalação de Dependências

In [ ]:
# Instalação das dependências necessárias para o laboratório
# Compatível com Python 3.11+ e Google Colab T4
!pip install -q ragas datasets langchain langchain-openai sentence-transformers opensearch-py langfuse pandas

print("Dependências instaladas com sucesso.")

---
## Célula 2 — Configuração de Variáveis de Ambiente

In [ ]:
import os

# ── OpenSearch ─────────────────────────────────────────────────────────────
os.environ["OPENSEARCH_HOST"] = os.getenv("OPENSEARCH_HOST", "localhost")
os.environ["OPENSEARCH_PORT"] = os.getenv("OPENSEARCH_PORT", "9200")
os.environ["OPENSEARCH_USER"] = os.getenv("OPENSEARCH_USER", "admin")
os.environ["OPENSEARCH_PASS"] = os.getenv("OPENSEARCH_PASS", "admin")

# ── vLLM ───────────────────────────────────────────────────────────────────
os.environ["VLLM_HOST"]     = os.getenv("VLLM_HOST",     "localhost")
os.environ["VLLM_PORT"]     = os.getenv("VLLM_PORT",     "8000")
os.environ["VLLM_BASE_URL"] = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
os.environ["VLLM_MODEL"]    = os.getenv("VLLM_MODEL",    "meta-llama/Meta-Llama-3.1-8B-Instruct")

# ── LangFuse ───────────────────────────────────────────────────────────────
os.environ["LANGFUSE_HOST"]       = os.getenv("LANGFUSE_HOST",       "http://localhost:3000")
os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY", "pk-lf-SUBSTITUA")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY", "sk-lf-SUBSTITUA")

# ── Modelo de Embedding ────────────────────────────────────────────────────
os.environ["EMBEDDING_MODEL"] = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")

# ── Constantes de uso local ─────────────────────────────────────────────────
VLLM_BASE_URL   = os.environ["VLLM_BASE_URL"]
VLLM_MODEL      = os.environ["VLLM_MODEL"]
EMBEDDING_MODEL = os.environ["EMBEDDING_MODEL"]
LANGFUSE_HOST   = os.environ["LANGFUSE_HOST"]

# Metas mínimas do syllabus
METAS = {
    "faithfulness":       0.80,
    "answer_relevancy":   0.75,
    "context_recall":     0.70,
    "context_precision":  0.70,
}

DATASET_LAB1 = "dataset_avaliacao_completo.json"

print("Variáveis de ambiente configuradas.")
print(f"  vLLM       : {VLLM_BASE_URL}")
print(f"  Modelo     : {VLLM_MODEL}")
print(f"  Embedding  : {EMBEDDING_MODEL}")
print(f"  LangFuse   : {LANGFUSE_HOST}")
print(f"\nMetas mínimas RAGAS:")
for metrica, meta in METAS.items():
    print(f"  {metrica:<22} : >= {meta}")

---
## Célula 3 — Inicialização das Conexões e Testes de Saúde

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langfuse import Langfuse
import warnings
warnings.filterwarnings("ignore")

# ── LLM Judge para RAGAS (via vLLM) ───────────────────────────────────────
llm_judge = ChatOpenAI(
    model=VLLM_MODEL,
    base_url=VLLM_BASE_URL,
    api_key="dummy",
    temperature=0.0,      # determinístico para avaliação
    max_tokens=1024,
)

VLLM_OK = False
try:
    resp_teste = llm_judge.invoke("Responda apenas: OK")
    VLLM_OK = True
    print(f"[OK] vLLM conectado — resposta: {resp_teste.content.strip()[:50]}")
except Exception as exc:
    print(f"[AVISO] vLLM não disponível: {exc}")
    print("  O RAGAS será executado em modo degradado.")

# ── Embeddings Judge para RAGAS ────────────────────────────────────────────
# Usa a API OpenAI-compatible do vLLM para embeddings
# Fallback: sentence-transformers local
embeddings_judge = OpenAIEmbeddings(
    model="BAAI/bge-m3",
    base_url=VLLM_BASE_URL,
    api_key="dummy",
    check_embedding_ctx_length=False,
)

EMBEDDINGS_OK = False
try:
    vetor_teste = embeddings_judge.embed_query("teste de embedding")
    EMBEDDINGS_OK = True
    print(f"[OK] Embeddings judge conectado — dim: {len(vetor_teste)}")
except Exception as exc:
    print(f"[AVISO] Embeddings via vLLM não disponíveis: {exc}")
    print("  Usando SentenceTransformer local como fallback.")
    from sentence_transformers import SentenceTransformer
    _encoder_local = SentenceTransformer(EMBEDDING_MODEL)

    # Wrapper simples para compatibilidade com RAGAS
    class EmbeddingLocal:
        def embed_query(self, text: str) -> list:
            return _encoder_local.encode(text, normalize_embeddings=True).tolist()
        def embed_documents(self, texts: list) -> list:
            return _encoder_local.encode(texts, normalize_embeddings=True).tolist()

    embeddings_judge = EmbeddingLocal()
    EMBEDDINGS_OK = True
    print(f"[OK] SentenceTransformer local carregado: {EMBEDDING_MODEL}")

# ── LangFuse ───────────────────────────────────────────────────────────────
LANGFUSE_OK = False
lf_client = None
try:
    lf_client = Langfuse(
        public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
        secret_key=os.environ["LANGFUSE_SECRET_KEY"],
        host=LANGFUSE_HOST,
    )
    lf_client.auth_check()
    LANGFUSE_OK = True
    print(f"[OK] LangFuse conectado em {LANGFUSE_HOST}")
except Exception as exc:
    print(f"[AVISO] LangFuse não disponível: {exc}")
    print("  Resultados serão salvos apenas em CSV.")

print(f"\nSaúde dos serviços:")
print(f"  vLLM      : {'OK' if VLLM_OK else 'INDISPONÍVEL'}")
print(f"  Embeddings: {'OK' if EMBEDDINGS_OK else 'INDISPONÍVEL'}")
print(f"  LangFuse  : {'OK' if LANGFUSE_OK else 'INDISPONÍVEL'}")

---
## Célula 4 — Carregamento do Dataset do LAB 1

In [ ]:
import json
import pandas as pd

dataset_pares: list[dict] = []

try:
    with open(DATASET_LAB1, "r", encoding="utf-8") as f:
        dataset_pares = json.load(f)
    print(f"[OK] Dataset carregado de '{DATASET_LAB1}': {len(dataset_pares)} pares")

except FileNotFoundError:
    print(f"[AVISO] Arquivo '{DATASET_LAB1}' não encontrado.")
    print("  Tentando regenerar a partir do LAB 1...")

    # Fallback: tenta carregar o corpus original
    try:
        DATASET_PATH_ORIG = "../datasets/corpus_avaliacao_aula5.json"
        with open(DATASET_PATH_ORIG, "r", encoding="utf-8") as f:
            corpus_orig = json.load(f)
        print(f"  Corpus original carregado: {len(corpus_orig)} pares")
        print("  ATENÇÃO: Execute o LAB 1 completo para gerar respostas reais.")
        print("  Usando apenas 10 pares com respostas placeholder para demonstração.")

        # Usa apenas 10 pares como fallback
        for par in corpus_orig[:10]:
            dataset_pares.append({
                "id":           par.get("id", ""),
                "question":     par["question"],
                "answer":       par.get("ground_truth", "")[:200],  # usa GT como proxy
                "contexts":     ["Contexto de fallback — execute o LAB 1 para contextos reais."],
                "ground_truth": par["ground_truth"],
                "tipo":         par.get("tipo", "nao_especificado"),
                "dificuldade":  par.get("dificuldade", "nao_especificada"),
            })

    except FileNotFoundError:
        print("  Corpus original também não encontrado. Criando 10 pares sintéticos.")
        tipos = ["processual_penal", "ambiental", "constitucional", "administrativo", "civil"]
        for i in range(10):
            tipo = tipos[i % len(tipos)]
            dataset_pares.append({
                "id":           f"q{i+1:03d}",
                "question":     f"Qual o procedimento previsto em lei para {tipo.replace('_', ' ')} no caso {i+1}?",
                "answer":       f"Conforme a legislação brasileira, o procedimento {i+1} para {tipo} implica observar os dispositivos legais pertinentes, garantindo o due process of law.",
                "contexts":     [f"Artigo {100+i} da Lei {8000+i}/2020: Dispõe sobre procedimentos de {tipo.replace('_', ' ')}."],
                "ground_truth": f"O procedimento correto para {tipo.replace('_', ' ')} está previsto no artigo {100+i} da Lei {8000+i}/2020, que estabelece requisitos específicos para casos desta natureza.",
                "tipo":         tipo,
                "dificuldade":  ["facil", "medio", "dificil"][i % 3],
            })

print(f"\nPares disponíveis para avaliação: {len(dataset_pares)}")
print(f"Campos por par: {list(dataset_pares[0].keys())}")

# Resumo do dataset
df = pd.DataFrame(dataset_pares)
if "tipo" in df.columns:
    print("\nDistribuição por tipo:")
    print(df["tipo"].value_counts().to_string())

---
## Célula 5 — Configuração do LLM Judge e Embeddings para RAGAS

In [ ]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# ── LLM Judge RAGAS ────────────────────────────────────────────────────────
# Usa o Llama 3.1 8B Instruct via vLLM como juiz avaliador
# temperature=0.0 garante avaliações determinísticas e reprodutíveis
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=VLLM_MODEL,
        base_url=VLLM_BASE_URL,
        api_key="dummy",
        temperature=0.0,
        max_tokens=1024,
    )
)

# ── Embeddings Judge RAGAS ─────────────────────────────────────────────────
# Usa o mesmo modelo BGE-M3 para calcular answer_relevancy via similaridade
try:
    ragas_embeddings = LangchainEmbeddingsWrapper(
        OpenAIEmbeddings(
            model="BAAI/bge-m3",
            base_url=VLLM_BASE_URL,
            api_key="dummy",
            check_embedding_ctx_length=False,
        )
    )
    # Teste rápido
    _ = ragas_embeddings.embed_query("teste")
    print("[OK] ragas_embeddings configurado via vLLM OpenAI-compatible")
except Exception as exc:
    print(f"[AVISO] Embeddings via vLLM falharam: {exc}")
    print("  Usando SentenceTransformer como fallback para embeddings RAGAS.")
    from sentence_transformers import SentenceTransformer
    from ragas.embeddings import BaseRagasEmbeddings
    from langchain_core.embeddings import Embeddings

    class STEmbeddingsWrapper(Embeddings):
        """Wrapper LangChain para SentenceTransformer, compatível com RAGAS."""
        def __init__(self, model_name: str):
            self._model = SentenceTransformer(model_name)

        def embed_documents(self, texts: list[str]) -> list[list[float]]:
            return self._model.encode(texts, normalize_embeddings=True).tolist()

        def embed_query(self, text: str) -> list[float]:
            return self._model.encode(text, normalize_embeddings=True).tolist()

    ragas_embeddings = LangchainEmbeddingsWrapper(STEmbeddingsWrapper(EMBEDDING_MODEL))
    print(f"[OK] ragas_embeddings configurado via SentenceTransformer ({EMBEDDING_MODEL})")

print("\nLLM Judge e Embeddings configurados para RAGAS.")
print(f"  LLM Judge  : {VLLM_MODEL}")
print(f"  Embeddings : {EMBEDDING_MODEL}")

---
## Célula 6 — Criação do Dataset HuggingFace para RAGAS

In [ ]:
from datasets import Dataset

# Prepara os dados no formato esperado pelo RAGAS
# RAGAS requer exatamente estes 4 campos:
#   - question     : str
#   - answer       : str
#   - contexts     : list[str]
#   - ground_truth : str

dados_ragas = {
    "question":     [p["question"]     for p in dataset_pares],
    "answer":       [p["answer"]       for p in dataset_pares],
    "contexts":     [p["contexts"]     for p in dataset_pares],
    "ground_truth": [p["ground_truth"] for p in dataset_pares],
}

# Cria o Dataset HuggingFace
hf_dataset = Dataset.from_dict(dados_ragas)

print(f"[OK] Dataset HuggingFace criado.")
print(f"  Número de pares    : {len(hf_dataset)}")
print(f"  Colunas            : {hf_dataset.column_names}")
print(f"  Tipo de 'contexts' : {type(hf_dataset[0]['contexts'])}")
print(f"\nExemplo do primeiro par:")
exemplo = hf_dataset[0]
print(f"  question    : {exemplo['question'][:100]}")
print(f"  answer      : {exemplo['answer'][:100]}")
print(f"  contexts[0] : {exemplo['contexts'][0][:100]}")
print(f"  ground_truth: {exemplo['ground_truth'][:100]}")

---
## Célula 7 — Execução da Avaliação RAGAS com as 4 Métricas

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

# Configura o LLM e embeddings nas métricas
METRICAS_RAGAS = [faithfulness, answer_relevancy, context_recall, context_precision]

for metrica in METRICAS_RAGAS:
    metrica.llm = ragas_llm
    if hasattr(metrica, "embeddings"):
        metrica.embeddings = ragas_embeddings

print("Iniciando avaliação RAGAS...")
print(f"  Número de pares  : {len(hf_dataset)}")
print(f"  Métricas         : faithfulness, answer_relevancy, context_recall, context_precision")
print(f"  LLM Judge        : {VLLM_MODEL}")
print()

resultados_ragas = None
erros_metricas: dict[str, str] = {}

# Execução com tratamento de exceções por métrica
try:
    resultados_ragas = evaluate(
        dataset=hf_dataset,
        metrics=METRICAS_RAGAS,
        raise_exceptions=False,  # captura erros por par sem abortar
    )
    print("[OK] Avaliação RAGAS concluída.")

except Exception as exc_geral:
    print(f"[AVISO] Erro na avaliação geral: {exc_geral}")
    print("  Tentando calcular métricas individualmente...")

    # Tenta cada métrica separadamente
    resultados_parciais: dict[str, float] = {}
    for metrica in METRICAS_RAGAS:
        nome = metrica.name
        try:
            res = evaluate(dataset=hf_dataset, metrics=[metrica], raise_exceptions=False)
            resultados_parciais[nome] = res[nome]
            print(f"  [OK] {nome}: {res[nome]:.4f}")
        except Exception as exc_m:
            erros_metricas[nome] = str(exc_m)
            resultados_parciais[nome] = float("nan")
            print(f"  [ERRO] {nome}: {exc_m}")

    # Cria objeto de resultado simulado
    class ResultadoSimulado:
        def __init__(self, scores: dict):
            self._scores = scores
        def __getitem__(self, key):
            return self._scores.get(key, float("nan"))
        def to_pandas(self):
            return pd.DataFrame([self._scores])

    resultados_ragas = ResultadoSimulado(resultados_parciais)

print("\nAvaliação concluída — prosseguindo para análise dos resultados.")

---
## Célula 8 — Tabela de Resultados: Valor Obtido vs. Meta

In [ ]:
import math

# Extrai os scores das 4 métricas
scores_obtidos: dict[str, float] = {}
for nome_metrica in METAS:
    try:
        valor = resultados_ragas[nome_metrica]
        scores_obtidos[nome_metrica] = float(valor) if not math.isnan(float(valor)) else 0.0
    except Exception:
        scores_obtidos[nome_metrica] = 0.0

# Exibe tabela de resultados
print("=" * 70)
print("RESULTADOS RAGAS — BASELINE NAIVE RAG (#T01)")
print("=" * 70)
print(f"{'Métrica':<25} {'Obtido':>8} {'Meta':>8} {'Status':>8}")
print("-" * 70)

resultados_tabela: list[dict] = []
todas_metas_atingidas = True

for metrica, meta in METAS.items():
    obtido = scores_obtidos.get(metrica, 0.0)
    atingiu = obtido >= meta
    status = "OK" if atingiu else "ABAIXO"
    emoji  = "[OK]" if atingiu else "[---]"
    if not atingiu:
        todas_metas_atingidas = False

    print(f"{metrica:<25} {obtido:>8.4f} {meta:>8.2f} {emoji:>8}")
    resultados_tabela.append({
        "metrica":    metrica,
        "obtido":     obtido,
        "meta":       meta,
        "atingiu":    atingiu,
        "diferenca":  round(obtido - meta, 4),
    })

print("-" * 70)
if todas_metas_atingidas:
    print("RESULTADO FINAL: Todas as metas atingidas!")
else:
    metricas_abaixo = [r["metrica"] for r in resultados_tabela if not r["atingiu"]]
    print(f"RESULTADO FINAL: {len(metricas_abaixo)} métrica(s) abaixo da meta: {', '.join(metricas_abaixo)}")
    print("  Consulte as próximas aulas para técnicas de melhoria.")
print("=" * 70)

---
## Célula 9 — Análise por Tipo Jurídico

In [ ]:
# Analisa Faithfulness médio por tipo jurídico
# Identifica quais tipos têm pior performance

# Obtém scores por par (se disponível no resultado RAGAS)
df_resultados = None
try:
    df_resultados = resultados_ragas.to_pandas()
    print(f"[OK] DataFrame de resultados por par gerado: {len(df_resultados)} linhas")
    print(f"  Colunas: {list(df_resultados.columns)}")
except Exception as exc:
    print(f"[AVISO] Não foi possível obter resultados por par: {exc}")
    print("  Gerando DataFrame sintético com scores globais.")
    # Cria um DataFrame com scores globais repetidos por par
    df_resultados = pd.DataFrame(dataset_pares)[["question", "tipo", "dificuldade"]].copy()
    for metrica in METAS:
        df_resultados[metrica] = scores_obtidos.get(metrica, 0.0)

# Adiciona metadados de tipo e dificuldade se não presentes
if "tipo" not in df_resultados.columns:
    df_tipos = pd.DataFrame(dataset_pares)[["tipo", "dificuldade"]]
    if len(df_tipos) == len(df_resultados):
        df_resultados = pd.concat([df_resultados.reset_index(drop=True),
                                   df_tipos.reset_index(drop=True)], axis=1)

print("\n" + "=" * 70)
print("ANÁLISE DE FAITHFULNESS POR TIPO JURÍDICO")
print("=" * 70)

if "tipo" in df_resultados.columns and "faithfulness" in df_resultados.columns:
    analise_tipo = (
        df_resultados
        .groupby("tipo")["faithfulness"]
        .agg(["mean", "min", "max", "count"])
        .sort_values("mean", ascending=True)  # piores primeiro
        .rename(columns={"mean": "média", "min": "mínimo", "max": "máximo", "count": "n_pares"})
    )

    print(f"\n{'Tipo Jurídico':<25} {'Média':>8} {'Mínimo':>8} {'Máximo':>8} {'N':>5}")
    print("-" * 70)
    for tipo, row in analise_tipo.iterrows():
        indicador = " <-- PIOR" if row["média"] < METAS["faithfulness"] else ""
        print(f"{tipo:<25} {row['média']:>8.4f} {row['mínimo']:>8.4f} {row['máximo']:>8.4f} {int(row['n_pares']):>5}{indicador}")

    # Identifica tipos com pior performance
    tipos_ruins = analise_tipo[analise_tipo["média"] < METAS["faithfulness"]]
    if not tipos_ruins.empty:
        print(f"\nTipos com Faithfulness abaixo de {METAS['faithfulness']}:")
        for tipo in tipos_ruins.index:
            print(f"  - {tipo}: {tipos_ruins.loc[tipo, 'média']:.4f}")
    else:
        print(f"\nTodos os tipos atingiram a meta de Faithfulness >= {METAS['faithfulness']}.")
else:
    print("Dados de tipo não disponíveis para análise por categoria.")
    print(f"  Faithfulness global: {scores_obtidos.get('faithfulness', 'N/A')}")

---
## Célula 10 — Registro dos Resultados no LangFuse

In [ ]:
# Registra métricas e metadados no LangFuse para rastreabilidade

if LANGFUSE_OK and lf_client is not None:
    try:
        # Cria trace principal do experimento
        trace = lf_client.trace(
            name="Aula5-LAB2-RAGAS-Baseline",
            metadata={
                "tecnica":    "Naive RAG (#T01)",
                "n_pairs":    len(dataset_pares),
                "modelo":     VLLM_MODEL,
                "embedding":  EMBEDDING_MODEL,
                "aula":       "Aula 5 — Avaliação RAGAS",
                "data":       "2026-04-17",
            },
            tags=["ragas", "baseline", "naive-rag", "aula5"],
        )

        # Registra um score por métrica
        for metrica, valor in scores_obtidos.items():
            lf_client.score(
                trace_id=trace.id,
                name=metrica,
                value=valor,
                comment=f"Meta: {METAS[metrica]} | {'ATINGIDA' if valor >= METAS[metrica] else 'ABAIXO'}",
            )
            print(f"[OK] Score registrado: {metrica} = {valor:.4f}")

        lf_client.flush()
        print(f"\n[OK] Trace registrado no LangFuse: 'Aula5-LAB2-RAGAS-Baseline'")
        print(f"  Trace ID: {trace.id}")
        print(f"  Acesse: {LANGFUSE_HOST}")

    except Exception as exc:
        print(f"[AVISO] Erro ao registrar no LangFuse: {exc}")
        print("  Continuando sem registro de observabilidade.")

else:
    print("[AVISO] LangFuse não disponível — pulando registro de observabilidade.")
    print(f"  Scores calculados localmente:")
    for metrica, valor in scores_obtidos.items():
        print(f"    {metrica}: {valor:.4f}")

---
## Célula 11 — Exportação dos Resultados por Par em CSV

In [ ]:
CAMINHO_CSV_RAGAS = "ragas_baseline_resultados.csv"

# Prepara DataFrame de resultados por par
colunas_resultado = ["question", "faithfulness", "answer_relevancy",
                     "context_recall", "context_precision"]

if df_resultados is not None:
    # Adiciona campos de metadados se disponíveis
    colunas_extras = [c for c in ["tipo", "dificuldade"] if c in df_resultados.columns]
    colunas_disponiveis = [c for c in colunas_resultado + colunas_extras
                           if c in df_resultados.columns]

    df_export = df_resultados[colunas_disponiveis].copy()

    # Adiciona coluna de status por par (se scores individuais disponíveis)
    if "faithfulness" in df_export.columns:
        df_export["faithfulness_ok"]      = df_export["faithfulness"]      >= METAS["faithfulness"]
    if "answer_relevancy" in df_export.columns:
        df_export["answer_relevancy_ok"]  = df_export["answer_relevancy"]  >= METAS["answer_relevancy"]
    if "context_recall" in df_export.columns:
        df_export["context_recall_ok"]    = df_export["context_recall"]    >= METAS["context_recall"]
    if "context_precision" in df_export.columns:
        df_export["context_precision_ok"] = df_export["context_precision"] >= METAS["context_precision"]

    df_export.to_csv(CAMINHO_CSV_RAGAS, index=False, encoding="utf-8")

    import os as _os
    tamanho = _os.path.getsize(CAMINHO_CSV_RAGAS) / 1024
    print(f"[OK] Resultados exportados: '{CAMINHO_CSV_RAGAS}' — {tamanho:.1f} KB")
    print(f"  Linhas: {len(df_export)} | Colunas: {list(df_export.columns)}")

    # Prévia das primeiras 3 linhas
    print("\nPrévia dos primeiros 3 resultados:")
    cols_preview = [c for c in ["question", "faithfulness", "answer_relevancy",
                                "context_recall", "context_precision"] if c in df_export.columns]
    preview = df_export[cols_preview].head(3).copy()
    if "question" in preview.columns:
        preview["question"] = preview["question"].str[:60] + "..."
    print(preview.to_string(index=False))

else:
    print("[AVISO] DataFrame de resultados não disponível. Exportando scores globais.")
    df_global = pd.DataFrame([{
        "nivel": "global",
        **scores_obtidos,
        **{f"{m}_meta": v for m, v in METAS.items()},
    }])
    df_global.to_csv(CAMINHO_CSV_RAGAS, index=False, encoding="utf-8")
    print(f"[OK] Scores globais exportados: '{CAMINHO_CSV_RAGAS}'")

---
## Célula 12 — Interpretação dos Resultados

### O que significam as métricas RAGAS?

#### Faithfulness (Fidelidade)
Mede se **cada afirmação da resposta gerada** pode ser inferida dos contextos recuperados. Um score baixo indica que o LLM está "alucinando" — produzindo informações não sustentadas pelos documentos da base de conhecimento jurídica. **Técnicas que podem melhorar este score:** Citation-aware RAG (Aula 6), Self-RAG com verificação de fatos (Aula 7), Corrective RAG (Aula 8).

#### Answer Relevancy (Relevância da Resposta)
Avalia se a resposta aborda diretamente a pergunta formulada, sem evasões ou informações tangenciais. Scores baixos podem indicar prompts mal construídos ou contextos recuperados que desviam o LLM do foco da pergunta. **Técnicas:** Query Rewriting (Aula 6), HyDE — Hypothetical Document Embeddings (Aula 7).

#### Context Recall (Cobertura do Contexto)
Verifica se os contextos recuperados **contêm as informações necessárias** para responder à pergunta (comparação com o ground truth). Um recall baixo indica que a busca kNN está falhando em recuperar os documentos mais importantes — problema de cobertura do índice ou de qualidade dos embeddings. **Técnicas:** Chunk Size Optimization, Hierarchical RAG, BM25 Hybrid Search (Aula 6).

#### Context Precision (Precisão do Contexto)
Mede se os contextos recuperados são **realmente relevantes** para a resposta (sem documentos ruidosos que confundem o LLM). Precisão baixa aumenta o risco de respostas divergentes. **Técnicas:** Reranking (Cross-encoder), MMR — Maximal Marginal Relevance (Aula 6), Contextual Compression (Aula 7).

### Roadmap de Melhoria por Aula

| Aula | Técnica | Métrica Alvo |
|------|---------|-------------|
| Aula 6 | Hybrid Search BM25 + kNN | Context Recall, Context Precision |
| Aula 6 | Reranking com Cross-encoder | Context Precision, Faithfulness |
| Aula 7 | HyDE — Hypothetical Document Embeddings | Answer Relevancy, Context Recall |
| Aula 7 | Self-RAG com CRITIC | Faithfulness |
| Aula 8 | Corrective RAG (CRAG) | Faithfulness, Context Precision |
| Aula 9 | CAG — Cache-Augmented Generation | Latência + todas as métricas |

> **Referência:** ES-JOLLY, S. et al. RAGAS: Automated Evaluation of Retrieval Augmented Generation. *arXiv:2309.15217*, 2023. Disponível em: <https://arxiv.org/abs/2309.15217>. Acesso em: abr. 2026.

---

## Checklist de Conclusão — LAB 2

Revise cada item antes de prosseguir para o **LAB 3 (Técnicas de Melhoria)**:

| # | Item | Status |
|---|------|--------|
| 1 | Dependências instaladas sem erros críticos | ☐ |
| 2 | Conexões com vLLM e LangFuse testadas e documentadas | ☐ |
| 3 | Dataset carregado (LAB 1) ou fallback de 10 pares ativado | ☐ |
| 4 | LLM Judge configurado com `temperature=0.0` para avaliação determinística | ☐ |
| 5 | Dataset HuggingFace criado com os 4 campos RAGAS | ☐ |
| 6 | Avaliação `evaluate()` executada com as 4 métricas | ☐ |
| 7 | Tabela comparativa (obtido vs. meta) exibida com indicadores | ☐ |
| 8 | Resultados registrados no LangFuse (trace + 4 scores) | ☐ |
| 9 | CSV `ragas_baseline_resultados.csv` exportado com scores por par | ☐ |

> **Nota ABNT:** Este laboratório segue as diretrizes da ABNT NBR 6023:2018 e ABNT NBR 14724:2011. Os resultados aqui obtidos constituem a **linha de base (#T01)** do projeto de avaliação comparativa, sendo referenciados nos relatórios das aulas subsequentes para mensuração de ganho de cada técnica RAG aplicada ao domínio jurídico e de segurança pública.